In [ ]:
# Block 1: Notebook Description
#Notebook description

#This notebook is being used to evaluate the techinical market conditions of a single asset and assess
#the appropriate strategy to take in order to maximize returns.

In [ ]:
# Block 2: Imports and Project Bootstrap
import logging
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()


from Quantapp.data import yf as qa_yf
from Quantapp.visualization import Plotter
from Quantapp.visualization.views.single_asset_profile.pricing.options_pricing import (
    plot_atm_iv_realized_view,
    plot_gbm_paths_view,
    plot_implied_volatility_by_strike_view,
    plot_iv_minus_realized_by_strike_view,
    plot_median_iv_minus_realized_view,
    plot_open_interest_overview_view,
    plot_open_interest_pot_ranges_view,
    plot_option_chain_table_view,
    plot_svi_surface_view,
)
from Quantapp.analytics import Helper, TimeSeriesAnalytics as Rolling
from Quantapp.data import (
    MacroDataClient,
    get_current_options_chain,
    get_historical_options_eod_panel,
    get_market_history,
)
from Quantapp.secrets import load_project_env, require_secret

load_project_env()

warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")

In [ ]:
# Block 3: set notebook parameters

options_params = {
    "ticker_str": "SOXL",
    "interval": "1d",
    "period": "10y",
    "risk_free_ticker": "^IRX",
    "risk_free_rate": 0.02 / 252,
    "time_frame_week": 7,
    "time_frame_short": 21,
    "time_frame_mid": 50,
    "time_frame_long": 200,
}

ticker_str = options_params["ticker_str"]
interval = options_params["interval"]
period = options_params["period"]
risk_free_ticker = options_params["risk_free_ticker"]
risk_free_rate = options_params["risk_free_rate"]
time_frame_week = options_params["time_frame_week"]
time_frame_short = options_params["time_frame_short"]
time_frame_mid = options_params["time_frame_mid"]
time_frame_long = options_params["time_frame_long"]

options_params

In [ ]:
# Massive API quick test: current options chain snapshot
test_ticker = ticker_str if 'ticker_str' in globals() else 'AAPL'
chain_test = get_current_options_chain(test_ticker)

chain_test_df = chain_test.chain
print(f"Ticker: {test_ticker} | Contracts returned: {len(chain_test_df)}")
chain_test_df

In [ ]:
# Block 3: TODO Notes
#take all compuation functions and put them in a separate file

#simplify the date x axis on the percent drawdown chart

#default the zoom range to a comfortable range, and create a dropdown to select the time range for Volatility section

#properly label and annoate the garch models

#Remove the VIX charting, its redudnant now that we have the volatility models

In [ ]:
# Block 4: Shared Clients

qc = Rolling()
qp = Plotter()
qe = MacroDataClient()
helper = Helper()

In [ ]:
# Block 6: Underlying Data Load
#Load data: underlying data
print(f"Loading data for {ticker_str} with period {period} and interval {interval}")
ticker_handle = qa_yf.Ticker(ticker_str)  # Used for options metadata/fallbacks through Quantapp.data.

market_history = get_market_history(
    symbols=[ticker_str],
    period=period,
    interval=interval,
    provider="yfinance",
    align=False,
)
ticker = market_history.get(str(ticker_str).strip().upper(), pd.DataFrame())

if ticker.empty:
    raise ValueError(f"No underlying price history returned for {ticker_str}.")

price_series = ticker['Close'].dropna()
log_returns = np.log(price_series / price_series.shift(1)).dropna()
spot_price = float(price_series.iloc[-1])

rolling_vol_window = min(len(log_returns), 252)
if rolling_vol_window < 2:
    raise ValueError(f"Not enough history to compute volatility for {ticker_str}.")

annualized_vol = log_returns.iloc[-rolling_vol_window:].std() * np.sqrt(252)

In [ ]:
# Block 7: Monte Carlo Simulation
#plot geometric brownian motion monte carlo simulation

# --- Parameters ---
lookback_days = 30 # Days to estimate volatility
projection_days = 30 # Days to simulate into the future
num_paths = 500 # Monte Carlo paths

# --- Function to calculate GBM Monte Carlo paths ---
def calculate_gbm_paths(prices, projection_days=30, lookback_days=30, num_paths=500, mu=None):
    recent_prices = prices[-lookback_days:]
    log_returns = np.log(recent_prices[1:] / recent_prices[:-1])
    sigma = log_returns.std()
    if mu is None:
        mu = 0

    S0 = prices[-1]
    T = projection_days
    N = num_paths
    dt = 1 / 252

    price_paths = np.zeros((T, N))
    price_paths[0, :] = S0

    for t in range(1, T):
        z = np.random.standard_normal(N)
        price_paths[t, :] = price_paths[t - 1, :] * np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z)

    median_path = np.median(price_paths, axis=1)
    p5 = np.percentile(price_paths, 5, axis=1)
    p95 = np.percentile(price_paths, 95, axis=1)

    return price_paths, median_path, p5, p95

# --- Main execution ---
gbm_history = price_series.tail(max(lookback_days + 1, 63))
gbm_prices = gbm_history.to_numpy()
gbm_dates = gbm_history.index.to_numpy()

# Calculate Monte Carlo paths
price_paths, median_path, p5, p95 = calculate_gbm_paths(
    gbm_prices,
    projection_days=projection_days,
    lookback_days=lookback_days,
    num_paths=num_paths
)

# Plot results
fig = plot_gbm_paths_view(gbm_dates, gbm_prices, price_paths, median_path, p5, p95, ticker_label=ticker_str)
fig.show()

In [ ]:
# Block 8: Expiration Dates
#Load data: Expiration Dates
print(f"Loading options expiration dates for {ticker_str}")
options_expiration_dates = pd.DataFrame(ticker_handle.options, columns=['Expiration Date'])

if options_expiration_dates.empty:
    raise ValueError(f"No listed option expirations returned for {ticker_str}.")

options_expiration_dates = options_expiration_dates.sort_values('Expiration Date').reset_index(drop=True)
options_expiration_dates['Date Till Expiration'] = (
    pd.to_datetime(options_expiration_dates['Expiration Date']) - pd.Timestamp.today().normalize()
).dt.days

In [ ]:
# Block 9B: Current Options Chain Snapshot
# Retrieve the current options chain snapshot.

print(f"Loading options chain for {ticker_str}")

call_contract_chain = {}
put_contract_chain = {}
drop_columns = ['lastTradeDate', 'contractSize', 'currency', 'percentChange', 'change']
today = pd.Timestamp.today().normalize()


try:
    massive_chain = get_current_options_chain(ticker_str, fallback_underlying_price=spot_price)
    massive_chain_df = massive_chain.chain
    underlying_price = massive_chain.underlying_price
    call_contract_chain = massive_chain.calls_by_expiration
    put_contract_chain = massive_chain.puts_by_expiration
    expirations = massive_chain.expirations
    options_expiration_dates = pd.DataFrame({'Expiration Date': expirations})
    options_expiration_dates['Date Till Expiration'] = (
        pd.to_datetime(options_expiration_dates['Expiration Date']) - today
    ).dt.days

    first_expiration_date = expirations[0]
    call_contracts = call_contract_chain[first_expiration_date].copy()
    put_contracts = put_contract_chain[first_expiration_date].copy()
    underlying_data = {'regularMarketPrice': underlying_price}
    first_expiration_chain = {'underlying': underlying_data, 'calls': call_contracts, 'puts': put_contracts}

    call_contract_chain_concat = massive_chain.calls
    put_contract_chain_concat = massive_chain.puts
    all_contracts_concat = pd.concat([call_contract_chain_concat, put_contract_chain_concat], ignore_index=True)

    print(f"Loaded options chain from Massive for {ticker_str}: {len(all_contracts_concat)} contracts")

except Exception as massive_error:
    print(f"Massive load failed ({massive_error}); falling back to Quantapp.data yfinance compatibility")

    first_expiration_date = options_expiration_dates['Expiration Date'].iloc[0]
    first_expiration_chain = ticker_handle.option_chain(first_expiration_date)
    underlying_data = first_expiration_chain.underlying
    call_contracts = first_expiration_chain.calls.copy()
    put_contracts = first_expiration_chain.puts.copy()

    for expiration_date in options_expiration_dates['Expiration Date']:
        option_chain = ticker_handle.option_chain(expiration_date)
        days_till_expiration = (pd.to_datetime(expiration_date) - today).days

        call_df = option_chain.calls.drop(columns=drop_columns, errors='ignore').copy()
        put_df = option_chain.puts.drop(columns=drop_columns, errors='ignore').copy()

        for df, option_type in ((call_df, 'Call'), (put_df, 'Put')):
            df['Days Till Expiration'] = days_till_expiration
            df['Expiration Date'] = expiration_date
            df['bid-ask spread'] = df['ask'] - df['bid']
            df['Expiration day'] = pd.to_datetime(expiration_date).day
            df['Expiration day name'] = pd.to_datetime(expiration_date).strftime('%A')
            df['Type'] = option_type
            df['mid'] = (df['bid'] + df['ask']) / 2

        call_contract_chain[expiration_date] = call_df
        put_contract_chain[expiration_date] = put_df

    expirations = sorted(call_contract_chain.keys())
    call_contract_chain_concat = pd.concat(call_contract_chain.values(), ignore_index=True)
    put_contract_chain_concat = pd.concat(put_contract_chain.values(), ignore_index=True)
    all_contracts_concat = pd.concat([call_contract_chain_concat, put_contract_chain_concat], ignore_index=True)

all_contracts_concat

In [ ]:
# Historical options chain EOD panel for the past 30 days (Massive).
# Output is one row per (as_of_date, contract) with EOD OHLCV fields.

HISTORICAL_LOOKBACK_DAYS = 30
MAX_CONTRACTS_PER_DAY = 12
MAX_DAYS_TO_EXPIRY = 120
MONEYNESS_BAND = 0.25

historical_options_panel = get_historical_options_eod_panel(
    ticker_str,
    underlying_history=ticker,
    lookback_days=HISTORICAL_LOOKBACK_DAYS,
    max_contracts_per_day=MAX_CONTRACTS_PER_DAY,
    max_days_to_expiry=MAX_DAYS_TO_EXPIRY,
    moneyness_band=MONEYNESS_BAND,
)
historical_chain_30d = historical_options_panel.chain
historical_chain_30d_summary = historical_options_panel.summary
print(
    f"Historical EOD rows: {len(historical_chain_30d)} across ",
    f"{historical_chain_30d['as_of_date'].nunique() if not historical_chain_30d.empty else 0} dates"
)
display(historical_chain_30d_summary)
historical_chain_30d

In [ ]:
# Block 10: Open Interest and PoT Ranges

if not expirations:
    raise ValueError('No option expirations are available to plot.')

fig = plot_open_interest_pot_ranges_view(
    call_contract_chain=call_contract_chain,
    put_contract_chain=put_contract_chain,
    expirations=expirations,
    spot_price=spot_price,
    annualized_vol=annualized_vol,
)
fig.show()

In [ ]:
# Block 11: Open Interest Overview

# Compute totals
total_oi_calls = [call_contract_chain[exp]['openInterest'].sum() for exp in expirations]
total_oi_puts = [put_contract_chain[exp]['openInterest'].sum() for exp in expirations]
dte_list = [call_contract_chain[exp]['Days Till Expiration'].iloc[0] for exp in expirations]

# Aggregate total OI (calls + puts)
total_oi_all = [c + p for c, p in zip(total_oi_calls, total_oi_puts)]

fig = plot_open_interest_overview_view(
    expirations=expirations,
    total_oi_calls=total_oi_calls,
    total_oi_puts=total_oi_puts,
    total_oi_all=total_oi_all,
    dte_list=dte_list,
)
fig.show()

In [ ]:
# Block 12: ATM IV and Realized Volatility
#plot the ATM IV and Realized Volatility

# Define ATM IV fetcher
def get_atm_iv_for_expiration(expiration_date, contract_chain):
    contracts = contract_chain.get(expiration_date)
    if contracts is None or contracts.empty:
        return np.nan

    required_columns = {'strike', 'impliedVolatility'}
    if not required_columns.issubset(contracts.columns):
        return np.nan

    valid_contracts = contracts[['strike', 'impliedVolatility']].copy()
    valid_contracts['strike'] = pd.to_numeric(valid_contracts['strike'], errors='coerce')
    valid_contracts['impliedVolatility'] = pd.to_numeric(valid_contracts['impliedVolatility'], errors='coerce')
    valid_contracts = valid_contracts.dropna(subset=['strike', 'impliedVolatility'])
    if valid_contracts.empty:
        return np.nan

    idx = (valid_contracts['strike'] - spot_price).abs().idxmin()
    return float(valid_contracts.loc[idx, 'impliedVolatility'])

# Calculate realized vol for each expiration
def get_realized_vol_for_expiration(expiration_date):
    days = (pd.to_datetime(expiration_date) - pd.Timestamp.today().normalize()).days
    if 1 < days < len(log_returns):
        window_returns = log_returns.iloc[-days:]
        realized_vol = window_returns.std() * np.sqrt(252)
        return realized_vol
    return np.nan

atm_df = pd.DataFrame({'Expiration Date': expirations})
atm_df['ATM IV Call'] = atm_df['Expiration Date'].apply(
    lambda exp: get_atm_iv_for_expiration(exp, call_contract_chain)
 )
atm_df['ATM IV Put'] = atm_df['Expiration Date'].apply(
    lambda exp: get_atm_iv_for_expiration(exp, put_contract_chain)
 )
atm_df['Days Till Expiration'] = atm_df['Expiration Date'].apply(
    lambda d: (pd.to_datetime(d) - pd.Timestamp.today().normalize()).days
 )
atm_df['Realized Vol'] = atm_df['Expiration Date'].apply(get_realized_vol_for_expiration)
atm_df = atm_df.sort_values('Days Till Expiration').reset_index(drop=True)

# Compute spreads
atm_df['IV-RV Call'] = atm_df['ATM IV Call'] - atm_df['Realized Vol']
atm_df['IV-RV Put'] = atm_df['ATM IV Put'] - atm_df['Realized Vol']

# Calculate Put-Call IV Skew
atm_df['IV Skew'] = atm_df['ATM IV Put'] - atm_df['ATM IV Call']

fig = plot_atm_iv_realized_view(atm_df, ticker_label=ticker_str)
fig.show()

In [ ]:
# Block 13: IV Minus Realized by Strike
#Plot IV - Realized Volatility by Strike for Calls and Puts

# Historical price data (must be a Series indexed by date, most recent last)
price_series = ticker['Close']
log_returns = np.log(price_series / price_series.shift(1))

# Spot price for reference line
spot_price = price_series.iloc[-1]
today = pd.to_datetime("today")

def build_iv_minus_realized_by_strike(contract_chain):
    iv_minus_realized_by_expiration = {}
    for exp in sorted(contract_chain.keys()):
        df_sorted = contract_chain[exp].copy().sort_values("strike")
        exp_date = pd.to_datetime(exp)
        days_till_exp = (exp_date - today).days

        if days_till_exp < 2 or days_till_exp > len(log_returns):
            continue

        realized_vol_n = log_returns.rolling(window=days_till_exp).std().iloc[-1] * np.sqrt(252)
        df_sorted["iv_minus_realized"] = df_sorted["impliedVolatility"] - realized_vol_n
        df_sorted["days_till_expiration"] = days_till_exp
        iv_minus_realized_by_expiration[exp] = df_sorted
    return iv_minus_realized_by_expiration

call_iv_minus_realized_by_expiration = build_iv_minus_realized_by_strike(call_contract_chain)
put_iv_minus_realized_by_expiration = build_iv_minus_realized_by_strike(put_contract_chain)

fig = plot_iv_minus_realized_by_strike_view(
    call_iv_minus_realized_by_expiration=call_iv_minus_realized_by_expiration,
    put_iv_minus_realized_by_expiration=put_iv_minus_realized_by_expiration,
    spot_price=spot_price,
)
fig.show()

In [ ]:
# Block 14: Median IV Minus Realized

today = pd.to_datetime("today")

# --- Helper function to calculate median IV - Realized Vol ---
def calc_median_iv_minus_realized(contract_chain, filter_func=None):
    median_dict = {}
    for exp in contract_chain.keys():
        df = contract_chain[exp].copy()
        if filter_func:
            df = df[filter_func(df)]
        if df.empty:
            continue
        exp_date = pd.to_datetime(exp)
        days_till_exp = (exp_date - today).days
        if days_till_exp < 2 or days_till_exp > len(log_returns):
            continue
        realized_vol_n = log_returns.rolling(window=days_till_exp).std().iloc[-1] * np.sqrt(252)
        median_val = (df['impliedVolatility'] - realized_vol_n).median()
        median_dict[exp_date] = median_val
    return median_dict

# --- Top subplot: All strikes ---
median_calls_all = calc_median_iv_minus_realized(call_contract_chain)
median_puts_all = calc_median_iv_minus_realized(put_contract_chain)

# --- Bottom subplot: OTM strikes ---
median_calls_otm = calc_median_iv_minus_realized(
    call_contract_chain,
    filter_func=lambda df: df['strike'] > spot_price
)
median_puts_otm = calc_median_iv_minus_realized(
    put_contract_chain,
    filter_func=lambda df: df['strike'] < spot_price
)

df_calls_all = pd.DataFrame({'Expiration': list(median_calls_all.keys()), 'Median': list(median_calls_all.values()), 'Type': 'Call'})
df_puts_all = pd.DataFrame({'Expiration': list(median_puts_all.keys()), 'Median': list(median_puts_all.values()), 'Type': 'Put'})
df_all = pd.concat([df_calls_all, df_puts_all])
df_all['DTE'] = (df_all['Expiration'] - today).dt.days

df_calls_otm = pd.DataFrame({'Expiration': list(median_calls_otm.keys()), 'Median': list(median_calls_otm.values()), 'Type': 'Call_OTM'})
df_puts_otm = pd.DataFrame({'Expiration': list(median_puts_otm.keys()), 'Median': list(median_puts_otm.values()), 'Type': 'Put_OTM'})
df_otm = pd.concat([df_calls_otm, df_puts_otm])
df_otm['DTE'] = (df_otm['Expiration'] - today).dt.days

fig = plot_median_iv_minus_realized_view(df_all, df_otm, today=today)
fig.show()

In [ ]:
# Block 15: SVI Surface
#plot SVI surface for calls and puts
from scipy.optimize import minimize
import scipy.interpolate as interp

print(f"Spot price for {ticker_str}: {spot_price}")

def svi_total_variance(k, a, b, rho, m, sigma):
    return a + b * (rho * (k - m) + np.sqrt((k - m) ** 2 + sigma ** 2))

def svi_objective(params, k, total_var):
    a, b, rho, m, sigma = params
    model_var = svi_total_variance(k, a, b, rho, m, sigma)
    return np.sum((model_var - total_var) ** 2)

def fit_svi(k, total_var):
    x0 = [0.1, 0.1, 0.0, 0.0, 0.1]
    bounds = [(-1, 1), (1e-5, 5), (-0.999, 0.999), (-5, 5), (1e-5, 5)]
    res = minimize(svi_objective, x0, args=(k, total_var), bounds=bounds, method='L-BFGS-B')
    return res.x if res.success else None

def get_realized_vol_surface(price_history, dtes):
    hist = price_history[['Close']].copy()
    hist['returns'] = np.log(hist['Close'] / hist['Close'].shift(1))
    hist.dropna(inplace=True)

    rv_data = []
    for dte in sorted(set(dtes)):
        dte = int(dte)
        sub_ret = hist['returns'].iloc[-dte:]
        if len(sub_ret) >= dte * 0.8:
            realized_vol = np.std(sub_ret) * np.sqrt(252)
            rv_data.append((dte, realized_vol))
    return pd.DataFrame(rv_data, columns=['Days Till Expiration', 'Realized Vol'])

call_df = call_contract_chain_concat.copy()
call_df = call_df[call_df['impliedVolatility'].notna() & (call_df['impliedVolatility'] > 0)].copy()
call_df['T'] = call_df['Days Till Expiration'] / 365.0

put_df = put_contract_chain_concat.copy()
put_df = put_df[put_df['impliedVolatility'].notna() & (put_df['impliedVolatility'] > 0)].copy()
put_df['T'] = put_df['Days Till Expiration'] / 365.0

def fit_svi_surface(df, spot_price):
    unique_Ts = np.sort(df['T'].unique())
    svi_params_per_T = {}

    for T in unique_Ts:
        slice_df = df[df['T'] == T]
        if len(slice_df) < 5:
            continue
        K = slice_df['strike'].values
        iv = slice_df['impliedVolatility'].values
        total_var = iv ** 2 * T
        k = np.log(K / spot_price)
        params = fit_svi(k, total_var)
        if params is not None:
            svi_params_per_T[T] = params

    Ts = np.array(sorted(svi_params_per_T.keys()))
    params_array = np.array([svi_params_per_T[T] for T in Ts])
    param_interpolators = [
        interp.interp1d(Ts, params_array[:, i], kind='linear', fill_value='extrapolate')
        for i in range(5)
    ]

    strike_grid = np.linspace(df['strike'].min(), df['strike'].max(), 50)
    T_grid = np.linspace(df['T'].min(), df['T'].max(), 30)
    iv_surface = np.full((len(T_grid), len(strike_grid)), np.nan)

    for i, T in enumerate(T_grid):
        a, b, rho, m, sigma = [f(T) for f in param_interpolators]
        k_vals = np.log(strike_grid / spot_price)
        total_var = svi_total_variance(k_vals, a, b, rho, m, sigma)
        iv_surface[i, :] = np.sqrt(total_var / T)

    return strike_grid, T_grid, iv_surface

call_strike_grid, call_T_grid, call_iv_svi_surface = fit_svi_surface(call_df, spot_price)
put_strike_grid, put_T_grid, put_iv_svi_surface = fit_svi_surface(put_df, spot_price)

all_dtes = np.unique(np.concatenate([
    call_df['Days Till Expiration'].unique(),
    put_df['Days Till Expiration'].unique()
]))

# Get RV surface from the shared top-loaded price history
rv_df = get_realized_vol_surface(ticker, all_dtes)

fig = plot_svi_surface_view(
    call_df=call_df,
    put_df=put_df,
    call_strike_grid=call_strike_grid,
    call_t_grid=call_T_grid,
    call_iv_svi_surface=call_iv_svi_surface,
    put_strike_grid=put_strike_grid,
    put_t_grid=put_T_grid,
    put_iv_svi_surface=put_iv_svi_surface,
    realized_vol_surface_df=rv_df,
    spot_price=spot_price,
    ticker_label=ticker_str,
)
fig.show()

In [ ]:
# Block 16: Positive Deviation Screen
import pandas as pd
import numpy as np
from scipy.interpolate import griddata

# Ensure all rows and columns are displayed
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

def calc_positive_deviation_otm(df, strike_grid, T_grid, iv_svi_surface, spot_price, strike_range=100, limit=100, option_type='call'):
    """
    Calculate OTM options where market IV exceeds SVI IV, with positive deviation and distance from spot.
    """
    # Prepare SVI points and values for interpolation
    points = []
    values = []
    for i, T in enumerate(T_grid):
        for j, K in enumerate(strike_grid):
            points.append((K, T))
            values.append(iv_svi_surface[i, j])
    points = np.array(points)
    values = np.array(values)

    # Market points for interpolation
    market_points = np.vstack([df['strike'].values, df['T'].values]).T

    # Interpolate SVI IV at market points
    svi_iv_at_market = griddata(points, values, market_points, method='linear')

    df = df.copy()
    df['svi_iv'] = svi_iv_at_market

    # Filter by strike distance from spot price
    df = df[(df['strike'] >= spot_price - strike_range) & (df['strike'] <= spot_price + strike_range)]

    # Filter for OTM contracts
    if option_type.lower() == 'call':
        df = df[df['strike'] > spot_price] # calls OTM
    elif option_type.lower() == 'put':
        df = df[df['strike'] < spot_price] # puts OTM

    # Filter where market IV > SVI IV
    df_filtered = df[df['impliedVolatility'] > df['svi_iv']].copy()

    # Calculate positive deviation
    df_filtered['positive_deviation'] = df_filtered['impliedVolatility'] - df_filtered['svi_iv']

    # Add distance from spot column
    df_filtered['distance_from_spot'] = df_filtered['strike'] - spot_price

    # Sort descending by positive deviation
    df_filtered_sorted = df_filtered.sort_values(by='positive_deviation', ascending=False)

    # Select relevant columns and limit to top N
    result_df = df_filtered_sorted[['strike', 'Days Till Expiration', 'impliedVolatility', 'svi_iv', 'positive_deviation', 'distance_from_spot']].head(limit)

    return result_df.reset_index(drop=True)


# --- Example Usage ---
calls_deviations_otm = calc_positive_deviation_otm(
    df=call_df,
    strike_grid=call_strike_grid,
    T_grid=call_T_grid,
    iv_svi_surface=call_iv_svi_surface,
    spot_price=spot_price,
    strike_range=100,
    limit=100,
    option_type='call'
)

puts_deviations_otm = calc_positive_deviation_otm(
    df=put_df,
    strike_grid=put_strike_grid,
    T_grid=put_T_grid,
    iv_svi_surface=put_iv_svi_surface,
    spot_price=spot_price,
    strike_range=100,
    limit=100,
    option_type='put'
)

print("Top 100 OTM Calls with Market IV > SVI IV (within Ã‚Â±100 strikes):")
display(calls_deviations_otm)

print("\nTop 100 OTM Puts with Market IV > SVI IV (within Ã‚Â±100 strikes):")
display(puts_deviations_otm)

In [ ]:
# Block 17: First Expiration Contracts
#retreive the contracts for the first expiration date and display them as a dataframe
columns_to_drop = ['currency', 'contractSize', 'percentChange', 'change']

call_contracts_table = call_contracts.drop(columns=columns_to_drop, errors='ignore').copy()
put_contracts_table = put_contracts.drop(columns=columns_to_drop, errors='ignore').copy()

call_contracts_table

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Get the current stock price from underlying data
current_price = underlying_data.get('regularMarketPrice', spot_price)

# Create a function to filter and display options based on strike range
def show_options_table(strikes_to_show):
    fig = plot_option_chain_table_view(
        call_contracts_table=call_contracts_table,
        put_contracts_table=put_contracts_table,
        current_price=current_price,
        strikes_to_show=strikes_to_show,
        expiration_date=first_expiration_date,
        ticker_label=ticker_str,
    )
    fig.show(config={'responsive': True, 'scrollZoom': True})

strike_dropdown = widgets.Dropdown(
    options=[(f"{i} Strikes", i) for i in [10, 20, 30, 40, 50, 60]],
    value=20,
    description='Show:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        clear_output(wait=True)
        display(strike_dropdown)
        show_options_table(change['new'])

strike_dropdown.observe(on_change, names='value')

display(strike_dropdown)
show_options_table(strike_dropdown.value)

In [ ]:
from datetime import datetime

# Reuse the shared top-loaded data instead of fetching a second ticker context.
ticker_symbol = ticker_str
current_price = float(underlying_data.get('regularMarketPrice', spot_price))
expiration_date = first_expiration_date
calls_df = call_contracts_table.copy()
puts_df = put_contracts_table.copy()

def decompose_contract_symbol(contract_symbol, underlying=ticker_symbol):
    try:
        prefix_len = len(underlying)
        expiration = f"20{contract_symbol[prefix_len:prefix_len+2]}-{contract_symbol[prefix_len+2:prefix_len+4]}-{contract_symbol[prefix_len+4:prefix_len+6]}"
        option_type = 'Call' if contract_symbol[prefix_len + 6] == 'C' else 'Put'
        strike_price = int(contract_symbol[prefix_len + 7:]) / 1000
        expiration_dt = datetime.strptime(expiration, '%Y-%m-%d')
        return underlying, expiration, option_type, strike_price, expiration_dt
    except Exception as e:
        print(f"Error decomposing symbol {contract_symbol}: {e}")
        return pd.Series([None, None, None, None, None])

calls_df[['Ticker', 'Expiration', 'OptionType', 'StrikePrice', 'ExpirationDate']] = calls_df['contractSymbol'].apply(
    lambda x: pd.Series(decompose_contract_symbol(x))
)
puts_df[['Ticker', 'Expiration', 'OptionType', 'StrikePrice', 'ExpirationDate']] = puts_df['contractSymbol'].apply(
    lambda x: pd.Series(decompose_contract_symbol(x))
)

current_date = datetime.now()
calls_df['DaysUntilExpiration'] = (calls_df['ExpirationDate'] - current_date).dt.days
puts_df['DaysUntilExpiration'] = (puts_df['ExpirationDate'] - current_date).dt.days

fig = plot_implied_volatility_by_strike_view(
    calls_df=calls_df,
    puts_df=puts_df,
    current_price=current_price,
    ticker_label=ticker_symbol,
    expiration_date=expiration_date,
)
fig.show()